# Single-segment intersection delay

Zoom in on one signal-bounded segment instead of the whole shape. We run the
same MIT-thesis §3.3 breakdown as `4_single_trip_delay_analysis.ipynb` (all
calculations live in `segment_delay.py`), then look at a single `SEGMENT_ID`
two ways:

1. **Every recorded trip overlaid** — distance into the segment over time and
   smoothed speed against position, so the segment-wide pattern (where buses
   slow and queue ahead of the downstream signal) is visible across all trips.
2. **Five sample trips in detail** — each trip's distance- and speed-over-time
   with stopping activities shaded by their guessed delay type (dwell /
   uniform / overflow / congestion), spanning the range of total delay.

In [ ]:
import geopandas as gpd
import pandas as pd
from IPython.display import display

from constants import CA_NAD83_Albers, CULVER_CITY_FEED_KEY, SERVICE_DATE, SHAPE_KEY_TO_SHAPE_ID_MAP
from _data_loaders import (
    get_culver_city_vehicle_positions,
    get_selected_shapes,
    get_traffic_signals,
    list_available_service_dates,
)
from segment_delay import analyze_trip_segment, run_segment_delay_analysis

## Configuration

`SHAPE_KEY` picks the route shape; `SEGMENT_ID` picks the signal-bounded
segment to drill into. Segment ids follow `segment_delay.build_segments`'
`sig<start>_to_sig<end>` convention (the signal ids are row order in the
signals GeoDataFrame). Use the segment table printed under *Run the breakdown*
to choose a valid id.

In [ ]:
SHAPE_KEY = "105"
SHAPE_ID = SHAPE_KEY_TO_SHAPE_ID_MAP[SHAPE_KEY]
SEGMENT_ID = "sig29_to_sig30"
print(f"Shape: {SHAPE_KEY} -> {SHAPE_ID}; segment: {SEGMENT_ID}")

## Load inputs

All trips for the shape across every service date, plus the shape geometry,
signals, and stops (with hand-curated near-side flags). Loading all dates reads
every per-date geoparquet, so this cell is the slow one.

In [ ]:
service_dates = list_available_service_dates()
vehicle_positions = pd.concat(
    [
        get_culver_city_vehicle_positions([SHAPE_KEY], service_date).assign(service_date=service_date)
        for service_date in service_dates
    ],
    ignore_index=True,
)
print(f"{len(vehicle_positions):,} positions over {len(service_dates)} dates")

shapes = get_selected_shapes(SERVICE_DATE, CULVER_CITY_FEED_KEY, [SHAPE_ID])
signals = get_traffic_signals()
stops = gpd.read_file(f"data/stops_{SHAPE_ID}.geojson").to_crs(CA_NAD83_Albers)

## Run the breakdown

Run the full §3.3 decomposition for the shape, then confirm the chosen
`SEGMENT_ID` exists and look at its geometry and daytime trip counts. The
breakdown is computed for every segment; the rest of the notebook just slices
to this one.

In [ ]:
result = run_segment_delay_analysis(vehicle_positions, shapes, signals, stops, SHAPE_ID)
assert SEGMENT_ID in result.segments.index, (
    f"{SEGMENT_ID} not found. Available segments:\n{list(result.segments.index)}"
)

segment = result.segments.loc[SEGMENT_ID]
display(segment.to_frame().T.round(1))
display(result.trip_counts.loc[[SEGMENT_ID]])

### Plot all distance and recorded speed on all pings from all recorded trips

Overlay every recorded trip (daytime and nighttime) that crossed this segment.
Each trip is re-smoothed and sliced to the segment with `analyze_trip_segment`;
speed is the smoothed speed from `smooth_trajectory.compute_speeds_per_trip`
(not the raw feed's `SPEED` field). Two views:

- **Left** — distance into the segment over elapsed time, one faint line per
  trip. Flat stretches are stops; the spread shows how variable progress is.
- **Right** — smoothed speed against position in the segment. The dashed line
  marks the downstream stop bar; speeds dropping toward zero as they approach
  it are the signature of signal/queue delay.

In [ ]:
# WORKING HERE
# we need to smooth all trips using the smooth_trajectory logic
# then we need to get all speeds
# then we need to subset to the segment range
# then plot every ping in the segment, maybe using a histogram
import matplotlib.pyplot as plt

from constants import MAX_SHAPE_JUMP_M, MAX_SNAP_DISTANCE_M
from match_shapes_vp import project_vp_on_shape
from smooth_trajectory import compute_speeds_per_trip, smooth_distances_per_trip

SEGMENT_BUFFER = 50
segment_start = segment.start_distance_m - SEGMENT_BUFFER
segment_end = segment.end_distance_m + SEGMENT_BUFFER
distance_along_shape_m = project_vp_on_shape(vehicle_positions, shapes, SHAPE_KEY_TO_SHAPE_ID_MAP, max_snap_distance=MAX_SNAP_DISTANCE_M, max_shape_jump=MAX_SHAPE_JUMP_M)
vp_smoothed = smooth_distances_per_trip(vehicle_positions, distance_along_shape_m, freq_seconds=1.0)
vp_speeds = compute_speeds_per_trip(vp_smoothed, freq_seconds=1.0)

In [ ]:
M_PER_S_TO_MPH = 2.236936

# Every recorded trip that crossed this segment (day and night), as
# (service_date, TRIP_KEY) pairs from the raw trip-segment table.
segment_trips = (
    result.trip_segment_table[result.trip_segment_table["segment_id"] == SEGMENT_ID]
    [["service_date", "TRIP_KEY"]]
    .drop_duplicates()
)
print(f"{len(segment_trips)} recorded trips through {SEGMENT_ID}")

# Re-smooth each trip and slice it to the segment (reuses analyze_trip_segment).
segment_trip_details = []
for trip in segment_trips.itertuples():
    try:
        detail = analyze_trip_segment(
            vehicle_positions, shapes, signals, stops, SHAPE_ID,
            trip.service_date, trip.TRIP_KEY, SEGMENT_ID,
        )
    except (ValueError, KeyError):
        continue  # too few pings to smooth, or trip never reaches the segment
    if len(detail.segment_trajectory) > 0:
        segment_trip_details.append(detail)

fig, (ax_distance, ax_speed) = plt.subplots(1, 2, figsize=(15, 5))
for detail in segment_trip_details:
    entry_time = detail.segment_trajectory.index[0]
    elapsed_s = (detail.segment_trajectory.index - entry_time).total_seconds()
    distance_into_segment_m = detail.segment_trajectory.to_numpy() - segment["start_distance_m"]
    speed_mph = detail.segment_speeds.to_numpy() * M_PER_S_TO_MPH

    ax_distance.plot(elapsed_s, distance_into_segment_m, color="tab:blue", lw=0.8, alpha=0.2)
    ax_speed.plot(distance_into_segment_m, speed_mph, color="tab:blue", lw=0.8, alpha=0.2)

ax_distance.set_xlabel("seconds since entering segment")
ax_distance.set_ylabel("distance into segment (m)")
ax_distance.set_title(f"{SEGMENT_ID}: distance over time")

ax_speed.axvline(segment["length_m"], color="black", lw=1, ls="--", label="downstream signal")
ax_speed.set_xlabel("distance into segment (m)")
ax_speed.set_ylabel("speed (mph)")
ax_speed.set_title(f"{SEGMENT_ID}: speed vs position")
ax_speed.legend()

fig.suptitle(f"{SEGMENT_ID} — {len(segment_trip_details)} trips", y=1.02)
fig.tight_layout()
plt.show()

### Analyze sample trips through the segment

Pick five daytime trips spanning the range of total delay (most-delayed to
least) and plot each one's distance- and speed-over-time, with stopping
activities shaded by their guessed delay type: **dwell** (gray), **uniform**
signal delay (blue), **overflow** (purple), **congestion** (orange). Black line
is the smoothed trajectory; red dots are the raw projected GPS pings. This is
the per-trip detail behind the aggregate overlay above — a sanity check on how
each stop got classified.

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

from segment_delay import DELAY_TYPE_COLORS


def plot_trip_segment(detail, title=None):
    """Distance- and speed-over-time for one trip in a segment.

    Black line = smoothed trajectory; red dots (distance plot) = raw projected
    GPS pings; shaded bands = stopping activities colored by delay type.
    """
    segment_entry_time = detail.segment_trajectory.index[0]
    elapsed_s = (detail.segment_trajectory.index - segment_entry_time).total_seconds()
    distance_into_segment_m = detail.segment_trajectory.to_numpy() - detail.segment["start_distance_m"]
    speed_mph = detail.segment_speeds.to_numpy() * M_PER_S_TO_MPH

    fig, (ax_distance, ax_speed) = plt.subplots(2, 1, sharex=True, figsize=(11, 7))
    ax_distance.plot(elapsed_s, distance_into_segment_m, color="black", lw=1.5, zorder=2)
    ax_speed.plot(elapsed_s, speed_mph, color="black", lw=1.5, zorder=2)

    # Raw projected GPS pings overlaid on the distance plot.
    raw_points = detail.trip_positions
    has_raw = len(raw_points) > 0
    if has_raw:
        raw_elapsed_s = (raw_points.index - segment_entry_time).total_seconds()
        ax_distance.scatter(
            raw_elapsed_s, raw_points.to_numpy() - detail.segment["start_distance_m"],
            s=16, color="tab:red", alpha=0.6, zorder=3,
        )

    present_types = set()
    for stop in detail.stops.itertuples():
        stop_start_s = (stop.start_timestamp - segment_entry_time).total_seconds()
        stop_end_s = (stop.end_timestamp - segment_entry_time).total_seconds()
        color = DELAY_TYPE_COLORS.get(stop.delay_type, "tab:gray")
        for ax in (ax_distance, ax_speed):
            ax.axvspan(stop_start_s, stop_end_s, color=color, alpha=0.35)
        present_types.add(stop.delay_type)

    ax_distance.set_ylabel("distance into segment (m)")
    ax_speed.set_ylabel("speed (mph)")
    ax_speed.set_xlabel("seconds since entering segment")
    ax_distance.set_title(title or str(detail.segment.name))

    legend_handles = [Line2D([0], [0], color="black", lw=1.5, label="smoothed")]
    if has_raw:
        legend_handles.append(
            Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="tab:red",
                   markeredgecolor="none", alpha=0.6, label="GPS pings")
        )
    legend_handles += [
        mpatches.Patch(color=DELAY_TYPE_COLORS[delay_type], alpha=0.35, label=delay_type)
        for delay_type in DELAY_TYPE_COLORS
        if delay_type in present_types
    ]
    ax_distance.legend(handles=legend_handles, loc="upper left", fontsize=8)
    fig.tight_layout()
    return fig

In [ ]:
import numpy as np

N_SAMPLES = 5

segment_daytime = (
    result.daytime_delays[result.daytime_delays["segment_id"] == SEGMENT_ID]
    .sort_values("total_delay_s", ascending=False)
    .reset_index(drop=True)
)
display(
    segment_daytime[
        ["service_date", "TRIP_KEY", "observed_travel_time_s",
         "uniform_delay_s", "overflow_delay_s", "congestion_delay_s", "total_delay_s"]
    ].round(1)
)

# Evenly spaced samples spanning highest -> lowest total delay.
sample_positions = np.linspace(0, len(segment_daytime) - 1, N_SAMPLES).round().astype(int)
sample_trips = segment_daytime.loc[np.unique(sample_positions)]

for trip in sample_trips.itertuples():
    detail = analyze_trip_segment(
        vehicle_positions, shapes, signals, stops, SHAPE_ID,
        trip.service_date, trip.TRIP_KEY, SEGMENT_ID,
    )
    plot_trip_segment(
        detail,
        title=f"{SEGMENT_ID} — {trip.service_date} trip {trip.TRIP_KEY} "
              f"(total delay {trip.total_delay_s:.0f}s)",
    )
    plt.show()

## Notes

Use the overlay to judge whether this segment's delay is concentrated at the
downstream stop bar (signal/queue) or spread along the segment (congestion),
and the sample trips to spot-check the dwell / uniform / overflow / congestion
classification against the actual trajectories. Disagreements here point back at
the thresholds in `segment_delay.py` (`SIGNAL_STOP_AREA_M`, `OVERFLOW_ZONE_M`,
`FASTER_PERIOD_SPEED_MPS`, the red-phase cap).